## Causal Inference
# School of Information, University of Michigan 
## Week 3

### Resources:
- Course Manual, which can be found in Coursera
- [Instrumental Variables & Randomized Encouragement Trials: Driving Engagement of Learners](MediumArticle.pdf)

## Part 1

### Background

Researchers in Coursera are interested in figuring out whether a certain learning style can cause a learner to be more engaged and thus more likely to ultimately complete a course.

### Data

The data file lecture3.csv contains 7 variables for 49,808 learners on the online learning platform Coursera. Below are the descriptions of each variable in the data:

- *id*: a unique identifier for each leaner
- *paid_enroll*: dummy variable that equals to 1 if a learner has paid for enrollment, 0 otherwise
- *prv_wk_nbr*: the most recent course week a learner has completed, as a measure of how far the learner is into the class (e.g. if a learner most recently completed week 2 of a course, this variable is equal to 2)
- *prv_wk_min*: the minutes a learner spent in the previous week on the platform
- *message*: equal to 1 if a learner is in the treatment group (i.e. he/she received a message), 0 otherwise
- *binge*: equal to 1 if a learner has binged, 0 otherwise (bingeing behavior is defined as completing and starting consecutive weeks of a course on the same day)
- *complete*: dummy variable that is equal to 1 if a learner completed the next week in the course, 0 otherwise


In [1]:
# Import statements. Run this cell.

import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
from scipy import stats
from linearmodels import IV2SLS
from mads.lib.path import assets

In [2]:
#Uploading data for assignment. Run this cell.
file = assets.find("lecture3.csv")
data_coursera = pd.read_csv(file)

#Uncomment below to see the first five lines of the dataframe.
data_coursera.head()

,id,paid_enroll,prv_wk_nbr,prv_wk_min,message,binge,complete
0,1,1,2,193,0,1,1
1,2,0,5,194,0,1,1
2,3,1,1,45,0,0,1
3,4,1,4,118,0,0,1
4,5,0,5,247,0,1,1


## Questions

We are interested in investigating whether bingeing, defined as completing and starting consecutive weeks of a course on the same day, increases the likelihood of completing the following week in a course.

**Note**: You can refer to the manual for the methods we use in the assignment if you need to. 

**Use the data_coursera dataframe uploaded above to answer the questions below unless otherwise specified.**

**1.** Using robust standard errors in the statsmodels module, regress the variable *complete* on *binge*. Assign the coefficient in front of *binge* to the variable `binge_coeff1_1` and ensure that its data type is float. (Round to four decimal places.) (1 pt)

In [5]:
# fit a linear probability model and use HC1 robust standard errors
reg1_1 = smf.ols("complete ~ binge", data = data_coursera).fit()
robust_reg1_1 = reg1_1.get_robustcov_results(cov_type = "HC1")

# binge is the second parameter because the first parameter is the intercept
binge_coeff1_1 = float(round(robust_reg1_1.params[1], 4))

print(f"binge coefficient: {binge_coeff1_1}")
robust_reg1_1.summary()

binge coefficient: 0.4619


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               complete   R-squared:                       0.258
Model:                            OLS   Adj. R-squared:                  0.258
Method:                 Least Squares   F-statistic:                     5416.
Date:                Sun, 21 Jun 2026   Prob (F-statistic):               0.00
Time:                        01:29:36   Log-Likelihood:                -4289.1
No. Observations:               49808   AIC:                             8582.
Df Residuals:                   49806   BIC:                             8600.
Df Model:                           1                                         
Covariance Type:                  HC1                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.4937      0.006     79.660      0.000       0.482       0.506
binge          0.4619      0.006     73.594      0.000       0.450       0.474
==============================================================================
Omnibus:                    18815.946   Durbin-Watson:                   1.989
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            86466.482
Skew:                          -1.810   Prob(JB):                         0.00
Kurtosis:                       8.344   Cond. No.                         5.36
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

In [6]:
# Hidden Tests, checking value of binge_coeff1_1.

**2.** Now run the regression (using robust standard errors) one more time with additional controls: *paid_enroll*, *prv_wk_nbr*, *prv_wk_min*. Assign the coefficient in front of *binge* to the variable `binge_coeff1_2` and ensure that its data type is float. (Round to four decimal places.) (1 pt)

In [7]:
# add observed learner characteristics that may confound bingeing and completion
reg1_2 = smf.ols(
    "complete ~ binge + paid_enroll + prv_wk_nbr + prv_wk_min",
    data = data_coursera,
).fit()
robust_reg1_2 = reg1_2.get_robustcov_results(cov_type = "HC1")

# extract the coefficient on binge and convert it to the required type
binge_index = reg1_2.model.exog_names.index("binge")
binge_coeff1_2 = float(round(robust_reg1_2.params[binge_index], 4))

print(f"controlled binge coefficient: {binge_coeff1_2}")
robust_reg1_2.summary()

controlled binge coefficient: 0.3172


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               complete   R-squared:                       0.334
Model:                            OLS   Adj. R-squared:                  0.334
Method:                 Least Squares   F-statistic:                     2629.
Date:                Sun, 21 Jun 2026   Prob (F-statistic):               0.00
Time:                        01:29:58   Log-Likelihood:                -1611.9
No. Observations:               49808   AIC:                             3234.
Df Residuals:                   49803   BIC:                             3278.
Df Model:                           4                                         
Covariance Type:                  HC1                                         
===============================================================================
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept       0.4317      0.007     63.185      0.000       0.418       0.445
binge           0.3172      0.007     47.510      0.000       0.304       0.330
paid_enroll     0.0073      0.002      3.174      0.002       0.003       0.012
prv_wk_nbr      0.0021      0.001      2.683      0.007       0.001       0.004
prv_wk_min      0.0007   1.18e-05     61.105      0.000       0.001       0.001
==============================================================================
Omnibus:                    11498.336   Durbin-Watson:                   1.990
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            32772.569
Skew:                          -1.217   Prob(JB):                         0.00
Kurtosis:                       6.142   Cond. No.                     1.28e+03
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
[2] The condition number is large, 1.28e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [8]:
# Hidden Tests, checking value of binge_coeff1_2.

When the point estimate we are interested in (i.e., the coefficient in front of variable binge) changes drastically with the inclusion of further covariates, we consider that to be worrisome for causal inference purposes (remember the regression sensitivity analysis). Furthermore, intuitively the positive correlation between bingeing and completion could just be the result of self-selection by learners who are both inherently more likely to complete as well as more likely to binge because of higher motivation. To overcome this problem, researchers in Coursera decided to run a randomized encouragement trial. They randomly split their learners into two groups. The treatment group received a message immediately after completing a week of material. The goal of the message was to encourage learners to start the next week right away (see below). The control group didn't receive the message.

<img src="Congratulations.png" alt="Treatment Message" style="width: 500px;"/>

## Part 2

### Questions 

We will be using the binary variable message as our instrument to investigate the impact of binging on completion of the following week’s lecture.

**1.** Since messages were randomly assigned, we know that the independence assumption is satisfied. What does the exclusion restriction mean in this context? (2 pts)

**Note**: This question will be manually graded. 

---
### STUDENT RESPONSE:

In this context, the exclusion restriction means that receiving a message affects completion of the following week’s lecture only through its effect on whether the student binges. The message must not directly influence next week’s completion or affect it through another channel, such as increased motivation, reminders, or general course engagement.

### Simple explanation

The **exclusion restriction** means that receiving a message can affect whether a student completes the following week’s lecture **only because the message changes whether the student binges**.

In other words:

> The message should not directly make students complete the next lecture. It should only affect completion indirectly by influencing binging behavior.

Here:

* **Instrument:** whether the student received a message
* **Treatment:** whether the student binged
* **Outcome:** whether the student completed the following week’s lecture

The required causal path is:

$$
\text{Message} \rightarrow \text{Binging} \rightarrow \text{Next-week completion}
$$

There should not be a separate direct path:

$$
\text{Message} \rightarrow \text{Next-week completion}
$$

---

### More in-depth explanation

The exclusion restriction requires that the randomly assigned message has **no effect on next week’s lecture completion except through its effect on binging**.

This assumption would be violated if the message influenced lecture completion through another mechanism. For example, the message might:

* motivate students to study more generally;
* remind them to complete future lectures;
* increase their engagement with the course;
* make them feel monitored or accountable.

In any of those cases, the message could affect next week’s completion even for students whose binging behavior did not change. We would then be unable to attribute the message’s effect on completion entirely to binging.

---

**2.** Let’s look at the first stage relationship.

**2a.** Using robust standard errors in the statsmodels module, regress variable *binge* on variable *message*. Assign the results (using the `.get_robustcov_results()` method) to the variable `robust_reg2_2a`. (1 pt)

In [31]:
# first stage: estimate whether message assignment changes bingeing behavior
first_stage = smf.ols("binge ~ message", data = data_coursera).fit()
robust_reg2_2a = first_stage.get_robustcov_results(cov_type = "HC1")

# with one instrument, the excluded-instrument F statistic equals t squared
message_index = first_stage.model.exog_names.index("message")
first_stage_t = robust_reg2_2a.tvalues[message_index]
first_stage_f = float(first_stage_t ** 2)

print(f"first-stage message coefficient: {robust_reg2_2a.params[message_index]:.6f}")
print(f"first-stage robust F statistic: {first_stage_f:.4f}")
robust_reg2_2a.summary()

first-stage message coefficient: 0.090336
first-stage robust F statistic: 911.9387


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  binge   R-squared:                       0.018
Model:                            OLS   Adj. R-squared:                  0.018
Method:                 Least Squares   F-statistic:                     911.9
Date:                Sun, 21 Jun 2026   Prob (F-statistic):          1.55e-198
Time:                        01:56:09   Log-Likelihood:                -16053.
No. Observations:               49808   AIC:                         3.211e+04
Df Residuals:                   49806   BIC:                         3.213e+04
Df Model:                           1                                         
Covariance Type:                  HC1                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.8243      0.002    342.078      0.000       0.820       0.829
message        0.0903      0.003     30.198      0.000       0.084       0.096
==============================================================================
Omnibus:                    18923.394   Durbin-Watson:                   2.007
Prob(Omnibus):                  0.000   Jarque-Bera (JB):            52775.181
Skew:                          -2.131   Prob(JB):                         0.00
Kurtosis:                       5.696   Cond. No.                         2.62
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

In [32]:
# Hidden Tests, checking the coefficients and standard errors of robust_reg2_2a.

**2b.** Do we have a strong first stage? Explain. (1 pt)

**Note**: This question will be manually graded.

---

### STUDENT RESPONSE:

The first-stage regression shows that receiving the message increased the probability of binging by approximately 0.0903, or 9.03 percentage points. The coefficient is statistically significant, with p<0.001, and its 95% confidence interval does not include zero. In addition, the robust first-stage F-statistic is approximately 911.94, which is far above the conventional threshold of 10. Therefore, the message strongly predicts binging and satisfies the relevance assumption for a valid instrumental variable.

---

### Simple explanation

This regression tests whether the randomly assigned **message** actually changes the probability that a student **binges**. This is called the **first-stage regression**.

The coefficient for `message` is:

$$
0.0903
$$

Because both `message` and `binge` are binary variables, this means:

> Students who received the message were about **9 percentage points more likely to binge** than students who did not receive the message.

The first-stage robust F-statistic is:

$$
F = 911.94
$$

This is far above the common rule-of-thumb threshold of 10. Therefore, `message` is a **strong and relevant instrument** for `binge`.

---

### More in-depth explanation

The first-stage model is approximately:

$$
\text{binge}_i = \alpha_0 + \alpha_1\text{message}_i + u_i
$$

The estimated intercept is:

$$
\hat{\alpha}_0 = 0.8243
$$

This means that among students who did **not** receive the message, the predicted probability of binging was approximately:

$$
82.43%
$$

The coefficient on `message` is:

$$
\hat{\alpha}_1 = 0.0903
$$

Therefore, receiving the message increased the predicted probability of binging by approximately:

$$
9.03 \text{ percentage points}
$$

The predicted binging rate for students who received the message is therefore approximately:

$$
0.8243 + 0.0903 = 0.9146
$$

or:

$$
91.46%
$$

The message coefficient is also statistically significant:

* ($t = 30.198$)
* ($p < 0.001$)
* 95% confidence interval: approximately ($0.084,\ 0.096$)

Because the confidence interval does not contain zero, the data provide very strong evidence that receiving the message affects binging behavior.

Most importantly for instrumental variables analysis, the robust first-stage F-statistic is approximately **911.94**. A commonly used guideline is that an F-statistic above 10 indicates that the instrument is not weak. Since 911.94 is much larger than 10, there is very strong evidence that `message` satisfies the **instrument relevance assumption**.

The low $(R^2)$ of 0.018 does not mean the instrument is weak. It means that the message explains only about 1.8% of the overall variation in binging. Instrument strength is assessed primarily using the first-stage coefficient and F-statistic, not the overall $(R^2)$.

---

**3.**  Let’s look at the intention-to-treat effect.

**3a.** Calculate the “intention-to-treat” (ITT) effect by running the reduced form regression. That is, using robust standard errors, regress *complete* on *message*. Based on your regression results, how much does receiving a message change the likelihood of completing the next week? Assign this number (the coefficient in front of variable *message*) to the variable `l_change2_3` and ensure that its data type is float. (Round to four decimal places.) (1 pt)

In [33]:
# reduced form: estimate the intention-to-treat effect of message assignment
reduced_form = smf.ols("complete ~ message", data = data_coursera).fit()
robust_reduced_form = reduced_form.get_robustcov_results(cov_type = "HC1")

message_index_rf = reduced_form.model.exog_names.index("message")
l_change2_3 = float(round(robust_reduced_form.params[message_index_rf], 4))
itt_pvalue2_3 = float(robust_reduced_form.pvalues[message_index_rf])

print(f"ITT effect: {l_change2_3}")
print(f"p-value: {itt_pvalue2_3:.6f}")
robust_reduced_form.summary()

ITT effect: 0.0113
p-value: 0.000036


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:               complete   R-squared:                       0.000
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     17.05
Date:                Sun, 21 Jun 2026   Prob (F-statistic):           3.65e-05
Time:                        01:56:15   Log-Likelihood:                -11725.
No. Observations:               49808   AIC:                         2.345e+04
Df Residuals:                   49806   BIC:                         2.347e+04
Df Model:                           1                                         
Covariance Type:                  HC1                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.8896      0.002    448.344      0.000       0.886       0.893
message        0.0113      0.003      4.129      0.000       0.006       0.017
==============================================================================
Omnibus:                    24479.796   Durbin-Watson:                   1.998
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           100302.901
Skew:                          -2.580   Prob(JB):                         0.00
Kurtosis:                       7.659   Cond. No.                         2.62
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity robust (HC1)
"""

In [34]:
# Hidden Tests, checking value of l_change2_3

**3b.** Based on the p-value, can you conclude that it is significant at the 5% level? Explain. (i.e. report BOTH the p-value AND the decision rule based on p-value to determine if the coefficient differs from 0 at the 5% significance level) (1 pt) 

**Note**: This question will be manually graded.

---

### STUDENT RESPONSE: 

The p-value for the message coefficient is approximately 0.000036. At the 5% significance level, the decision rule is to reject the null hypothesis $H_0: \beta_{message} = 0$ when the p-value is less than 0.05. Since $0.000036 < 0.05$, we reject the null hypothesis and conclude that the intention-to-treat effect is statistically significant at the 5% level. Receiving the message increased the probability of completing the following week’s lecture by approximately 1.13 percentage points.

---

### Simple explanation

The p-value for `message` is:

$$
[p = 0.000036]
$$

To test significance at the 5% level, compare the p-value with:

$$
[\alpha = 0.05]
$$

The decision rule is:

* If (p < 0.05), reject the null hypothesis.
* If (p \geq 0.05), fail to reject the null hypothesis.

Because:

$$
[0.00036 < 0.05]
$$

we **reject the null hypothesis** and conclude that the ITT effect is statistically significant at the 5% level.

---

### More in-depth explanation

The hypotheses are:

$$
H_0:\beta_{\text[{message}}=0
$$

$$
H_A:\beta_{\text{mes[sage}}\neq 0
$$

The null hypothesis says that receiving a message has no effect on the probability of completing the following week’s lecture.

The estimated coefficient for `message` is
$$
[0.0113
]$$

This means that students assigned to receive the message were approximately **1.13 percentage points more likely to complete the following week’s lecture** than students who were not assigned the message.

The p-value is approximately:
[$
0.000036
]$$

Because this is much smaller than 0.05, the observed coefficient would be very unlikely if the true effect of the message were zero. Therefore, we reject the null hypothesis and conclude that the message coefficient differs significantly from zero.

The 95% confidence interval is approximately:

$$
[0.006,\ 0.017]
$$

Because this interval does not contain zero, it supports the same conclusion.

---

**4.** The ITT doesn’t take into account that some users may not comply with the treatment assignment. With heterogeneous treatment effects, we have an additional assumption: monotonicity. This means that there are “no defiers” in the population. 
Explain what “no defiers” means in this context and what its implications are for the identification of treatment effects.
(2 pts)

**Note**: This question will be manually graded.

---

### STUDENT RESPONSE:

“No defiers” means that there are no users who would binge when they do not receive the message but would stop binging when they do receive it. The message may increase a user’s likelihood of binging or leave it unchanged, but it cannot decrease it. Under monotonicity, along with the other instrumental-variable assumptions, the IV estimate identifies the Local Average Treatment Effect: the causal effect of binging on next-week lecture completion for compliers, meaning users whose binging behavior is changed by the message. Without this assumption, the behavior of compliers and defiers could offset each other, and the treatment effect could not be cleanly identified.

---

### Simple explanation

“No defiers” means there are no students who would behave in the **opposite direction** of the message assignment.

In this setting, a defier would be someone who:

* would binge if they **did not** receive the message, but
* would not binge if they **did** receive the message.

The monotonicity assumption says the message may encourage some students to binge, or have no effect on others, but it does not cause anyone to become less likely to binge.

---

### More in-depth explanation

Let:

* $(D_i(1))$ = whether student (i) would binge if assigned the message
* $(D_i(0))$ = whether student (i) would binge if not assigned the message

Monotonicity requires:

$
D_i(1) \geq D_i(0)
$

for every student.

This allows the following types of students:

* **Compliers:** binge when messaged, but not when unmessaged
* **Always-takers:** binge regardless of the message
* **Never-takers:** do not binge regardless of the message

It rules out:

* **Defiers:** binge only when they do not receive the message

This assumption matters because the instrumental-variable estimate identifies the **Local Average Treatment Effect**, or LATE: the causal effect of binging on next-week lecture completion for the students whose binging behavior was changed by the message—the compliers.

Without monotonicity, the effect among compliers could be mixed with the opposite behavior of defiers. The observed first-stage effect would then represent the difference between the proportion of compliers and defiers, making it impossible to cleanly interpret the IV estimate as the treatment effect for compliers.

---

**5.** Assuming the no defiers assumption is satisfied,  calculate the share of “always-takers,” which is given by the probability of bingeing when assigned to not receive a message. (You can calculate this value by dividing the number of learners who binged without receiving a message to the total number of learners in the no message group.) Assign the value to the variable `at_share2_5` and ensure that its data type is float. (Round to four decimal places.) (1 pt)

In [35]:
# always-takers binge even when they are assigned to the no-message group
no_message = data_coursera.loc[data_coursera["message"] == 0, "binge"]
at_share2_5 = float(round(no_message.mean(), 4))

print(f"always-taker share: {at_share2_5}")

always-taker share: 0.8243


In [36]:
# Hidden Tests, checking value of at_share2_5

**6.** Similarly, calculate the share of “never-takers”, which is given by the probability of not bingeing when assigned to receiving a message. Assign the value to the variable `nt_share2_6` and ensure that its data type is float. (Round to four decimal places.) (1 pt)

In [37]:
# never-takers do not binge even when they are assigned to receive the message
message_group = data_coursera.loc[data_coursera["message"] == 1, "binge"]
nt_share2_6 = float(round((message_group == 0).mean(), 4))

print(f"never-taker share: {nt_share2_6}")

never-taker share: 0.0854


In [38]:
# Hidden Tests, checking value of nt_share2_6.

**7.** ITT effects divided by the difference in compliance rates between the two groups (i.e. the effect of the instrument on the treatment) captures the causal effect of bingeing on compliers who binged as a result of the experiment. That is, the IV estimate we are interested in is equal to the reduced form divided by the first stage. Calculate the IV estimate manually, that is, divide the reduced form coefficient from Part 2, Question 3a (rounded to four decimal places) by the first stage coefficient from Part 2, Question 2a (rounded to four decimal places). Assign the value to the variable `answer2_7` and ensure that its data type is float. (Round to four decimal places.) (1 pt)

In [39]:
# use the assignment's required rounded reduced-form and first-stage coefficients
first_stage_coeff_rounded = float(
    round(robust_reg2_2a.params[message_index], 4)
)

# the Wald/IV estimate is the reduced-form effect divided by the first-stage effect
answer2_7 = float(round(l_change2_3 / first_stage_coeff_rounded, 4))

print(f"manual IV estimate: {answer2_7}")

manual IV estimate: 0.1251


In [40]:
# Hidden Tests, checking value of answer2_7.

**8.** In order to obtain a measure of the precision of our IV estimate we want to use the 2SLS method. Using robust standard errors, run a two-stage least squares regression, where the outcome variable is *complete*, the instrumented variable is *binge* and the instrument is the variable *message*. Use the IV2SLS module from the linearmodels library. Assign the results using the `.fit()` method to the variable `iv2sls2_8`. (2 pts)

**Note**: Be sure to remove any NAs from the dataframe before proceeding. 

In [41]:
# retain complete cases for the outcome, treatment, and instrument
iv_data = data_coursera[["complete", "binge", "message"]].dropna().copy()

# 2SLS instruments binge with randomized message assignment and reports robust SEs
iv2sls2_8 = IV2SLS.from_formula(
    "complete ~ 1 + [binge ~ message]",
    data=iv_data,
).fit(cov_type="robust")

iv2sls2_8.summary

<class 'linearmodels.compat.statsmodels.Summary'>
"""
                          IV-2SLS Estimation Summary                          
==============================================================================
Dep. Variable:               complete   R-squared:                      0.1213
Estimator:                    IV-2SLS   Adj. R-squared:                 0.1213
No. Observations:               49808   F-statistic:                    19.395
Date:                Sun, Jun 21 2026   P-value (F-stat)                0.0000
Time:                        01:56:28   Distribution:                  chi2(1)
Cov. Estimator:                robust                                         
                                                                              
                             Parameter Estimates                              
==============================================================================
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      0.7862     0.0248     31.690     0.0000      0.7376      0.8348
binge          0.1254     0.0285     4.4040     0.0000      0.0696      0.1812
==============================================================================

Endogenous: binge
Instruments: message
Robust Covariance (Heteroskedastic)
Debiased: False
"""

In [42]:
# Hidden Tests, checking the coefficients and standard errors of iv2sls2_8.